# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.



## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```


Мои предсказания и объяснения:

1. {'A': 2, 'B': 1} Метод r.get(c, 0) берёт текущее значение (или 0, если ключа нет) и прибавляет 1
2. [0, 1, 1] Функция f изменяет исходный список a напрямую через метод .append(), добавляя единицу при каждом вызове.
3. 2 Цикл сначала переводит строки в нижний регистр, в итоге остаются только уникальные "almaty" и "astana"
4.  {'ab': 2, 'abc': 3} Словарь не может содержать одинаковых ключей, поэтому при втором проходе ключ "ab" просто перезаписывается новым значением длины (2)
5. 5 Функция печатает 5, но не имеет инструкции return, поэтому неявно возвращает None, которое присваивается переменной x и выводится.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [7]:
#1
def to_num(s):
    try:
        return float(s)
    except ValueError:
        return 0.0

def clean_city(s):
    if not s:
        return ""
    s = s.strip().lower()
    mapping = {
        'almaty': 'Алматы', 'алматы': 'Алматы',
        'astana': 'Астана', 'астана': 'Астана',
        'shymkent': 'Шымкент', 'шымкент': 'Шымкент',
        'актобе': 'Актобе', 'aktobe': 'Актобе',
        'атырау': 'Атырау', 'atyrau': 'Атырау',
        'караганда': 'Караганда', 'karaganda': 'Караганда'
    }
    return mapping.get(s, s.capitalize())

In [8]:
#2
cat_count = {}
for r in rows:
    cat = r['category']
    cat_count[cat] = cat_count.get(cat, 0) + 1

print("2. Заказы по категориям:")
for cat, cnt in sorted(cat_count.items(), key=lambda x: x[1], reverse=True):
    print(f"{cat}: {cnt}")

2. Заказы по категориям:
Одежда: 1529
Электроника: 1185
Дом: 1105
Красота: 991
Книги: 653


In [11]:
# 3
rev = {}
cnt = {}
for r in rows:
    if r['status'] == 'оплачен':
        cat = r['category']
        val = to_num(r['amount'])
        rev[cat] = rev.get(cat, 0) + val
        cnt[cat] = cnt.get(cat, 0) + 1

print("\n3. Выручка и средний чек:")
for cat in rev:
    avg = rev[cat] / cnt[cat] if cnt.get(cat, 0) > 0 else 0
    print(f"{cat:<15} Выручка: {rev[cat]:>10.1f} | Средний чек: {avg:>8.1f}")



3. Выручка и средний чек:
Одежда          Выручка: 16754010.0 | Средний чек:  12149.4
Красота         Выручка:  4973320.0 | Средний чек:   5588.0
Дом             Выручка:  9665680.0 | Средний чек:   9793.0
Книги           Выручка:  2125160.0 | Средний чек:   3645.2
Электроника     Выручка: 62785340.0 | Средний чек:  58953.4


In [14]:
# 4
month_count = {}
for r in rows:
    # Защита от пустых дат
    if not r.get('order_date'): continue
    m = r['order_date'][:7]
    month_count[m] = month_count.get(m, 0) + 1

print("\n4. Заказы по месяцам:")
for m, c in month_count.items():
    print(f"{m}: {c}")


4. Заказы по месяцам:
2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [16]:
# Стандарт
#5
m_rev = {}
for r in rows:
    if r.get('order_date') and r['status'] == 'оплачен':
        m = r['order_date'][:7]
        m_rev[m] = m_rev.get(m, 0) + to_num(r['amount'])

print("5. Выручка по месяцам:")
for m, r_val in m_rev.items():
    print(f"{m}: {r_val}")

pct_change = ((m_rev.get('2026-10', 0) - m_rev.get('2026-09', 0)) / m_rev.get('2026-09', 1)) * 100
print(f"Изменение октября к сентябрю: {pct_change:.1f}%")

5. Выручка по месяцам:
2026-09: 54677220.0
2026-10: 41626290.0
Изменение октября к сентябрю: -23.9%


In [17]:
#6
canc = {}
tot = {}
for r in rows:
    city = clean_city(r.get('city', ''))
    if not city: continue
    tot[city] = tot.get(city, 0) + 1
    if r['status'] == 'отменен':
        canc[city] = canc.get(city, 0) + 1

print("\n6. Доля отменённых заказов:")
for city in tot:
    percent = (canc.get(city, 0) / tot[city]) * 100
    print(f"{city}: {percent:.1f}%")


6. Доля отменённых заказов:
Алматы: 0.0%
Астана: 0.0%
Караганда: 0.0%
Шымкент: 0.0%
Атырау: 0.0%
Актобе: 0.0%


#7
На первом месте клиент с ID TEST. С ним «не так» то, что это очевидно системный тестовый аккаунт. Его выручка достигает почти 30 миллионов, что является аномальным выбросом в данных, который критически искажает общую финансовую статистику за сентябрь.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж